In [ ]:
from cosipy.nonimaging.bgo.chi2_localization import BGOLocalizerChi2
import numpy as np

In [ ]:
# Inizializza con i tre LUT (pickle) e nside
#TODO download them from wasabi
dir_path = "/data/test_newrepo/"
soft_name = "LUT_soft_lut.npy"
medium_name = "LUT_medium_lut.npy"
hard_name = "LUT_hard_lut.npy"

localizer = BGOLocalizerChi2(
    soft_lut_path=dir_path+soft_name,
    medium_lut_path=dir_path+medium_name,
    hard_lut_path= dir_path+hard_name,
    nside=32,
)

In [ ]:
def ra_dec_to_theta_phi(ra, dec):

    theta = 90-dec
    
    phi = ra
    
    return theta, phi
def spherical_to_radec_deg(theta_deg, phi_deg):

    dec = 90.0 - theta_deg
    ra = phi_deg % 360.0
    return ra, dec


In [ ]:
# Counts by panel name; the localizer reorders them for its LUTs

true_ra,true_dec = spherical_to_radec_deg(84.021,49.922)

s_counts = {
    "ACS-C-X0": 2356.0, "ACS-C-X1": 2431.0,
    "ACS-B-Y0": 3116.0, "ACS-B-Y1": 2863.0,
    "ACS-A-Z0": 1924.0, "ACS-A-Z1": 1951.0,
}
b_counts = {
    "ACS-C-X0": 1079.35, "ACS-C-X1": 1131.83,
    "ACS-B-Y0": 1277.07, "ACS-B-Y1": 1323.91,
    "ACS-A-Z0": 1098.75, "ACS-A-Z1": 1089.84,
}

In [ ]:
from astropy.coordinates import SkyCoord
import astropy.units as u
from scoords import Attitude, SpacecraftFrame
ori_file  = "/home/cosi/cosi/data/background/dc4/DC3_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.ori"
data = np.loadtxt(ori_file, usecols=(1, 2, 3, 4, 5, 6, 7, 8), delimiter=' ', skiprows=1, comments=("#", "EN"))

In [ ]:

time_grb = 1835517732.9649906

tempi = data[:, 0]

idx = np.argmin(np.abs(tempi - time_grb))

nearest_row = data[idx]

print("Index:", idx)
print("Nearest ori bin:", tempi[idx])
print("Row", nearest_row)

i = idx+1

In [ ]:
x_pointing = SkyCoord(data[:, 2][i]*u.deg, data[:, 1][i]*u.deg, frame='galactic')
z_pointing = SkyCoord(data[:, 4][i]*u.deg, data[:, 3][i]*u.deg, frame='galactic')
attitude = Attitude.from_axes(x=x_pointing, z=z_pointing, frame='galactic')

result = localizer.localize(s_counts, b_counts,attitude=attitude)
print(result)